In [ ]:
!pip install scipy scikit-learn matplotlib pandas numpy

In [5]:
import json
import numpy as np
import pickle
from pathlib import Path
from sentence_transformers import SentenceTransformer
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import pandas as pd

# ============================================================
# 1. LOAD DATA
# ============================================================

def load_hashtag_data(file_path: str) -> dict:
    """Load hashtag data từ file JSON"""
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

# ============================================================
# 2. EMBEDDING VỚI BGE-M3 (BATCH)
# ============================================================

def embed_texts_batch(
    texts: list[str],
    model: SentenceTransformer,
    batch_size: int = 64
) -> np.ndarray:
    """Embed texts theo batch để tăng tốc"""

    all_embeddings = []

    for i in range(0, len(texts), batch_size):

        batch = texts[i:i + batch_size]

        print(f"  Embedding batch {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}")

        embeddings = model.encode(
            batch,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)

def create_and_save_embeddings(
    data: dict,
    output_path: str,
    batch_size: int = 32
) -> dict:
    """Tạo embeddings và lưu file với đầy đủ thông tin"""

    print("Loading BGE-M3 model...")

    model = SentenceTransformer('BAAI/bge-m3')

    hashtags = list(data.keys())
    texts = list(data.values())

    print(f"Embedding {len(texts)} texts...")

    embeddings = embed_texts_batch(texts, model, batch_size)

    result = {
        'hashtags': hashtags,
        'texts': texts,
        'embeddings': embeddings,
        'mapping': data
    }

    with open(output_path, 'wb') as f:
        pickle.dump(result, f)

    print(f"Saved embeddings to {output_path}")

    return result

def load_embeddings(file_path: str) -> dict:
    """Load embeddings đã lưu"""
    with open(file_path, 'rb') as f:
        return pickle.load(f)

# ============================================================
# 3. HIERARCHICAL CLUSTERING
# ============================================================

def run_hierarchical_clustering(
    embeddings: np.ndarray,
    n_clusters_list: list[int],
    method: str = 'ward'
) -> tuple[np.ndarray, dict]:
    """
    Chạy hierarchical clustering với nhiều số clusters
    Returns: linkage_matrix, {n_clusters: labels}
    """

    print("Computing linkage matrix...")

    if method == 'ward':
        # Ward cần dùng trực tiếp trên data
        linkage_matrix = linkage(embeddings, method='ward')
    else:
        # Các method khác dùng distance matrix
        distances = pdist(embeddings, metric='cosine')
        linkage_matrix = linkage(distances, method=method)

    cluster_results = {}

    for n_clusters in n_clusters_list:
        print(f"  Cutting tree for {n_clusters} clusters...")
        labels = fcluster(linkage_matrix, n_clusters, criterion='maxclust')
        cluster_results[n_clusters] = labels

    return linkage_matrix, cluster_results

def save_cluster_results(
    hashtags: list[str],
    texts: list[str],
    cluster_results: dict,
    output_dir: str
):
    """Lưu kết quả clustering ra files để đánh giá"""

    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)

    for n_clusters, labels in cluster_results.items():
        # Tạo DataFrame
        df = pd.DataFrame({
            'hashtag': hashtags,
            'text': texts,
            'cluster': labels
        })

        # Sort theo cluster
        df = df.sort_values('cluster')

        # Lưu CSV
        csv_path = output_path / f'clusters_{n_clusters}.csv'
        df.to_csv(csv_path, index=False, encoding='utf-8')

        # Lưu summary theo cluster
        summary_path = output_path / f'clusters_{n_clusters}_summary.txt'
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write(f"=== HIERARCHICAL CLUSTERING: {n_clusters} CLUSTERS ===\n\n")

            for cluster_id in sorted(df['cluster'].unique()):
                cluster_items = df[df['cluster'] == cluster_id]
                f.write(f"\n--- Cluster {cluster_id} ({len(cluster_items)} items) ---\n")

                for _, row in cluster_items.iterrows():
                    f.write(f"  {row['hashtag']}: {row['text']}\n")

        print(f"Saved results for {n_clusters} clusters")

# ============================================================
# 4. PCA VISUALIZATION
# ============================================================

def visualize_clusters_pca(
    embeddings: np.ndarray,
    cluster_results: dict,
    texts: list[str],
    output_dir: str
):
    """Trực quan hóa clusters bằng PCA 2D"""

    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)

    print("Applying PCA...")
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(embeddings)

    print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}")

    # Vẽ cho mỗi số clusters
    for n_clusters, labels in cluster_results.items():
        fig, ax = plt.subplots(figsize=(14, 10))

        scatter = ax.scatter(
            embeddings_2d[:, 0],
            embeddings_2d[:, 1],
            c=labels,
            cmap='tab20',
            alpha=0.7,
            s=100
        )

        # Thêm labels cho một số điểm
        if len(texts) <= 50:
            for i, txt in enumerate(texts):
                ax.annotate(
                    txt[:15] + '...' if len(txt) > 15 else txt,
                    (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                    fontsize=8,
                    alpha=0.8
                )

        ax.set_title(f'Hierarchical Clustering - {n_clusters} Clusters\n(PCA 2D Projection)')
        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')

        plt.colorbar(scatter, label='Cluster')
        plt.tight_layout()

        fig_path = output_path / f'pca_clusters_{n_clusters}.png'
        plt.savefig(fig_path, dpi=150)
        plt.close()

        print(f"Saved PCA plot for {n_clusters} clusters")

# ============================================================
# 5. MAIN EXECUTION
# ============================================================

def main():
    # Cấu hình
    INPUT_FILE = '/content/drive/MyDrive/tikharm_detection/datasets/output/hashtag_map.json'          # File JSON input
    EMBEDDINGS_FILE = 'embeddings.pkl'    # File lưu embeddings
    OUTPUT_DIR = '/content/drive/MyDrive/tikharm_detection/datasets/output'        # Thư mục output
    BATCH_SIZE = 64
    N_CLUSTERS_LIST = [20, 50, 100, 200]

    # Load data
    print("=" * 50)
    print("STEP 1: Loading data")
    print("=" * 50)
    data = load_hashtag_data(INPUT_FILE)
    print(f"Loaded {len(data)} hashtags")

    # Embedding
    print("\n" + "=" * 50)
    print("STEP 2: Creating embeddings")
    print("=" * 50)

    if Path(EMBEDDINGS_FILE).exists():
        print(f"Loading existing embeddings from {EMBEDDINGS_FILE}")
        embed_data = load_embeddings(EMBEDDINGS_FILE)
    else:
        embed_data = create_and_save_embeddings(
            data, EMBEDDINGS_FILE, BATCH_SIZE
        )

    print(f"Embeddings shape: {embed_data['embeddings'].shape}")

    # Clustering
    print("\n" + "=" * 50)
    print("STEP 3: Hierarchical Clustering")
    print("=" * 50)

    linkage_matrix, cluster_results = run_hierarchical_clustering(
        embed_data['embeddings'],
        N_CLUSTERS_LIST,
        method='ward'
    )

    # Save linkage matrix
    np.save(f'{OUTPUT_DIR}/linkage_matrix.npy', linkage_matrix)

    # Save cluster results
    print("\n" + "=" * 50)
    print("STEP 4: Saving results")
    print("=" * 50)

    save_cluster_results(
        embed_data['hashtags'],
        embed_data['texts'],
        cluster_results,
        OUTPUT_DIR
    )

    # Visualization
    print("\n" + "=" * 50)
    print("STEP 5: PCA Visualization")
    print("=" * 50)

    visualize_clusters_pca(
        embed_data['embeddings'],
        cluster_results,
        embed_data['texts'],
        OUTPUT_DIR
    )

    print("\n" + "=" * 50)
    print("DONE!")
    print("=" * 50)

if __name__ == '__main__':
    main()


STEP 1: Loading data
Loaded 1355 hashtags

STEP 2: Creating embeddings
Loading BGE-M3 model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding 1355 texts...
  Embedding batch 1/22
  Embedding batch 2/22
  Embedding batch 3/22
  Embedding batch 4/22
  Embedding batch 5/22
  Embedding batch 6/22
  Embedding batch 7/22
  Embedding batch 8/22
  Embedding batch 9/22
  Embedding batch 10/22
  Embedding batch 11/22
  Embedding batch 12/22
  Embedding batch 13/22
  Embedding batch 14/22
  Embedding batch 15/22
  Embedding batch 16/22
  Embedding batch 17/22
  Embedding batch 18/22
  Embedding batch 19/22
  Embedding batch 20/22
  Embedding batch 21/22
  Embedding batch 22/22
Saved embeddings to embeddings.pkl
Embeddings shape: (1355, 1024)

STEP 3: Hierarchical Clustering
Computing linkage matrix...
  Cutting tree for 20 clusters...
  Cutting tree for 50 clusters...
  Cutting tree for 100 clusters...
  Cutting tree for 200 clusters...

STEP 4: Saving results
Saved results for 20 clusters
Saved results for 50 clusters
Saved results for 100 clusters
Saved results for 200 clusters

STEP 5: PCA Visualization
Applying PCA...
PCA

In [9]:
import pandas as pd
from collections import defaultdict


# ==========================================
# 1. LOAD HASHTAG DISTRIBUTION
# ==========================================

def load_hashtag_distribution(txt_file):

    hashtag_label_freq = defaultdict(lambda: defaultdict(int))

    current_hashtag = None
    reading_section = False

    with open(txt_file, "r", encoding="utf-8") as f:

        for line in f:

            line = line.rstrip()

            # bắt đầu section
            if "===== HASHTAG DISTRIBUTION ACROSS LABELS =====" in line:
                reading_section = True
                continue

            # dừng section
            if "===== HASHTAGS ONLY APPEAR IN ONE LABEL =====" in line:
                break

            if not reading_section:
                continue

            if not line.strip():
                continue

            # hashtag line
            if not line.startswith(" "):
                current_hashtag = line.strip().lower()
                continue

            # label line
            label, count = line.split(":")
            label = label.strip()
            count = int(count.strip())

            hashtag_label_freq[current_hashtag][label] += count

    return hashtag_label_freq


# ==========================================
# 2. LOAD CLUSTER CSV
# ==========================================

def load_cluster_csv(csv_file):

    df = pd.read_csv(csv_file)

    df["hashtag"] = df["hashtag"].str.replace("#", "").str.lower()

    return df


# ==========================================
# 3. COMPUTE CLUSTER LABEL RATIO
# ==========================================

def compute_cluster_ratios(df, hashtag_label_freq):

    cluster_counts = defaultdict(lambda: defaultdict(int))

    for _, row in df.iterrows():

        hashtag = row["hashtag"]
        cluster = row["cluster"]

        if hashtag not in hashtag_label_freq:
            continue

        for label, freq in hashtag_label_freq[hashtag].items():
            cluster_counts[cluster][label] += freq

    results = []

    for cluster, label_counts in cluster_counts.items():

        total = sum(label_counts.values())

        row = {"cluster": cluster}

        for label, count in label_counts.items():
            row[label] = count / total

        results.append(row)

    df_result = pd.DataFrame(results).fillna(0)

    return df_result


# ==========================================
# 4. SAVE RESULT
# ==========================================

def save_cluster_distribution(df, output_file):

    df = df.sort_values("cluster")

    df.to_csv(output_file, index=False)

    print("Saved:", output_file)


# ==========================================
# RUN
# ==========================================

cluster_csv = "/content/drive/MyDrive/tikharm_detection/datasets/output/clusters_200.csv"
hashtag_stats_txt = "/content/drive/MyDrive/tikharm_detection/datasets/output/hashtag_statistics.txt"
output_csv = "/content/drive/MyDrive/tikharm_detection/datasets/output/cluster_label_distribution_200.csv"

freq_data = load_hashtag_distribution(hashtag_stats_txt)

cluster_df = load_cluster_csv(cluster_csv)

result_df = compute_cluster_ratios(cluster_df, freq_data)

save_cluster_distribution(result_df, output_csv)

Saved: /content/drive/MyDrive/tikharm_detection/datasets/output/cluster_label_distribution_200.csv


In [17]:
# meta_cluster_and_pca.py
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering, KMeans

# -----------------------------
# Config
# -----------------------------
INPUT_CSV = "/content/drive/MyDrive/tikharm_detection/datasets/output/cluster_label_distribution_20.csv"  # file từ bước trước
N_META_CLUSTERS = 5                          # số meta-clusters mong muốn
OUTPUT_DIR = "/content/drive/MyDrive/tikharm_detection/datasets/output/meta_cluster_output/20_clusters_5"
CLUSTERING_METHOD = "agglomerative"           # "agglomerative" | "kmeans"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# -----------------------------
# 1) Load CSV & preprocess
# -----------------------------
def load_and_prepare(input_csv):
    df = pd.read_csv(input_csv)
    # ensure 'cluster' column present
    if 'cluster' not in df.columns:
        raise ValueError("Input CSV must contain a 'cluster' column")

    # sort by cluster for stable ordering
    df = df.sort_values('cluster').reset_index(drop=True)

    # extract features (all columns except 'cluster')
    feature_cols = [c for c in df.columns if c != 'cluster']
    X = df[feature_cols].fillna(0).astype(float).values

    # keep cluster ids for later annotation
    cluster_ids = df['cluster'].values

    return df, cluster_ids, feature_cols, X


# -----------------------------
# 2) Standardize features
# -----------------------------
def standardize_features(X):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    return Xs, scaler


# -----------------------------
# 3) Meta-clustering
# -----------------------------
def meta_cluster(Xs, n_meta=N_META_CLUSTERS, method="agglomerative"):
    if method == "agglomerative":
        model = AgglomerativeClustering(n_clusters=n_meta,metric="euclidean",linkage="ward")
        labels = model.fit_predict(Xs)
    elif method == "kmeans":
        model = KMeans(n_clusters=n_meta, n_init=10, random_state=42)
        labels = model.fit_predict(Xs)
    else:
        raise ValueError("Unknown method: choose 'agglomerative' or 'kmeans'")

    return labels, model


# -----------------------------
# 4) PCA for visualization
# -----------------------------
def pca_2d(Xs):
    pca = PCA(n_components=2, random_state=42)
    X2 = pca.fit_transform(Xs)
    explained = pca.explained_variance_ratio_
    return X2, pca, explained


# -----------------------------
# 5) Save results
# -----------------------------
def save_results(df_orig, cluster_ids, feature_cols, meta_labels, X2, explained, outdir):
    # DataFrame with meta-cluster
    out_df = df_orig.copy()
    out_df['meta_cluster'] = meta_labels
    out_csv = Path(outdir) / "cluster_meta_mapping.csv"
    out_df.to_csv(out_csv, index=False, encoding='utf-8')
    print("Saved meta mapping:", out_csv)

    # Save PCA scatter plot
    fig, ax = plt.subplots(figsize=(12, 9))
    scatter = ax.scatter(X2[:, 0], X2[:, 1], c=meta_labels, cmap='tab20', s=120, alpha=0.8)
    ax.set_title(f"PCA (2D) of clusters — meta-clusters: {len(np.unique(meta_labels))}\n"
                 f"Explained var: PC1 {explained[0]:.2%}, PC2 {explained[1]:.2%}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

    # annotate cluster ids (optional: annotate only if few points)
    # annotate only first N if too many
    annotate = True
    max_annotate = 200
    if X2.shape[0] <= max_annotate and annotate:
        for i, cid in enumerate(cluster_ids):
            ax.annotate(str(cid), (X2[i, 0], X2[i, 1]), fontsize=8, alpha=0.9)

    plt.colorbar(scatter, ax=ax, label='meta-cluster')
    plt.tight_layout()
    fig_path = Path(outdir) / "pca_meta_clusters.png"
    plt.savefig(fig_path, dpi=200)
    plt.close()
    print("Saved PCA plot:", fig_path)

    # Also save a summary per meta-cluster: mean proportions and counts
    summary_rows = []
    for m in sorted(np.unique(meta_labels)):
        mask = out_df['meta_cluster'] == m
        subset = out_df.loc[mask, feature_cols]
        mean_props = subset.mean().to_dict()
        count = mask.sum()
        row = {'meta_cluster': m, 'count': int(count)}
        row.update(mean_props)
        summary_rows.append(row)
    summary_df = pd.DataFrame(summary_rows).fillna(0)
    summary_csv = Path(outdir) / "meta_cluster_summary.csv"
    summary_df.to_csv(summary_csv, index=False, encoding='utf-8')
    print("Saved meta-cluster summary:", summary_csv)


# -----------------------------
# Main flow
# -----------------------------
def main():
    df, cluster_ids, feature_cols, X = load_and_prepare(INPUT_CSV)
    Xs, scaler = standardize_features(X)
    meta_labels, model = meta_cluster(Xs, n_meta=N_META_CLUSTERS, method=CLUSTERING_METHOD)
    X2, pca, explained = pca_2d(Xs)
    save_results(df, cluster_ids, feature_cols, meta_labels, X2, explained, OUTPUT_DIR)
    print("Done.")


if __name__ == "__main__":
    main()

Saved meta mapping: /content/drive/MyDrive/tikharm_detection/datasets/output/meta_cluster_output/20_clusters_5/cluster_meta_mapping.csv
Saved PCA plot: /content/drive/MyDrive/tikharm_detection/datasets/output/meta_cluster_output/20_clusters_5/pca_meta_clusters.png
Saved meta-cluster summary: /content/drive/MyDrive/tikharm_detection/datasets/output/meta_cluster_output/20_clusters_5/meta_cluster_summary.csv
Done.
